# Term Structure Modelling via Cox-Ingersoll-Ross: Calibration, Extensions & Out-of-Sample Forecasting
## Module Overview
This notebook implements and stress-tests the **Cox-Ingersoll-Ross (CIR)** framework for modelling the dynamics of zero-coupon yield curves. The pipeline spans the full quantitative workflow:
1. **Robust data engineering** — business-day alignment, cross-sectional spline repair, and outlier filtering via Rolling MAD.
2. **Cross-sectional calibration** — Tenor-Weighted MSE minimised with a global Differential Evolution solver under strict Feller-condition constraints.
3. **Three structural extensions** — Affine Jump-Diffusion (AJD/Duffie-Pan-Singleton), CIR++ deterministic shift (Brigo-Mercurio), and Two-Factor CIR with algebraic state extraction.
4. **Blind out-of-sample backtest** — frozen parameters, forward-stepping Kalman Filter, and per-tenor $R^2$ / RMSE diagnostics.

### The Instantaneous Short Rate SDE
The CIR short rate $r_t$ satisfies:
$$dr_t = \kappa(\theta - r_t)\,dt + \sigma\sqrt{r_t}\,dW_t$$

| Symbol | Role |
|--------|------|
| $\kappa$ | Mean-reversion speed |
| $\theta$ | Long-run equilibrium rate |
| $\sigma$ | Volatility coefficient |
| $W_t$ | Standard Brownian motion |

Strict positivity of $r_t$ is guaranteed by the **Feller condition**:
$$2\kappa\theta > \sigma^2$$

## 0. Environment Setup & Library Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import differential_evolution, minimize, NonlinearConstraint
from scipy.interpolate import CubicSpline
from scipy import integrate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ── Colour palette ────────────────────────────────────────────────────────────
PALETTE = {
    'actual'    : '#0d3b66',
    'predicted' : '#ee6c4d',
    'theta'     : '#3d9970',
    'anchor'    : '#e63946',
}
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True,
                     'grid.alpha': 0.35, 'font.size': 10})

# ── Tenor universe ────────────────────────────────────────────────────────────
RAW_COLS     = ['ZC025YR','ZC050YR','ZC075YR','ZC100YR',
                'ZC200YR','ZC500YR','ZC1000YR','ZC2000YR','ZC3000YR']
TENOR_LABELS = ['3M','6M','9M','1Y','2Y','5Y','10Y','20Y','30Y']
TENOR_YEARS  = [0.25, 0.5, 0.75, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0]
COL_TO_TENOR = dict(zip(RAW_COLS, TENOR_YEARS))
LABEL_TO_COL = dict(zip(TENOR_LABELS, RAW_COLS))

TRAIN_CSV = '/content/drive/MyDrive/Quant_Project/train_data.csv'
TEST_CSV  = '/content/drive/MyDrive/Quant_Project/test_data.csv'

print('Environment ready.')
print(f'numpy {np.__version__} | pandas {pd.__version__}')

## 1. Data Engineering & Preprocessing
### Dataset Schema
Nine zero-coupon bond yields are available daily:
| Column | Tenor | Label |
|--------|-------|-------|
| ZC025YR | 0.25 yr | 3M |
| ZC050YR | 0.50 yr | 6M |
| ZC075YR | 0.75 yr | 9M |
| ZC100YR | 1.00 yr | 1Y |
| ZC200YR | 2.00 yr | 2Y |
| ZC500YR | 5.00 yr | 5Y |
| ZC1000YR | 10.00 yr | 10Y |
| ZC2000YR | 20.00 yr | 20Y |
| ZC3000YR | 30.00 yr | 30Y |

> Yields are stored in **decimal form** throughout (e.g. 0.05 = 5%).

### Cleaning Pipeline (4 stages)
**Stage 1 — Cross-Sectional Spline Repair:**
Zeros and isolated NaNs across the tenor axis on a single date are treated as missing quotes and repaired with a `cubicspline` fitted across the available tenors. This avoids dropping entire rows without introducing any temporal look-ahead.

**Stage 2 — Business-Day Calendar Alignment:**
The index is forced onto a strict Mon-Fri calendar via `.asfreq('B')`. All resulting gaps (weekends, holidays) are resolved with **forward-fill only**, never interpolation across time, to prevent look-ahead bias.

**Stage 3 — 1 bp Floor:**
The CIR diffusion term $\sigma\sqrt{r_t}$ is undefined at $r_t \leq 0$. A hard lower bound of 1 basis point (0.0001) is applied dataset-wide.

**Stage 4 — Rolling MAD Outlier Filter:**
A 20-day rolling Median Absolute Deviation filter (MAD, not standard deviation) flags single-day anomalies more than 5× the local spread and replaces them with the rolling median. MAD is robust to the very extreme values that standard deviation would amplify.

In [ ]:
class YieldPipeline:
    """
    End-to-end data cleaning pipeline for zero-coupon yield panel data.
    Stages: load → spline-repair gaps → calendar align → MAD filter.
    """
    def __init__(self, filepath: str):
        self.filepath  = filepath
        self.raw       = None
        self.clean     = None

    # ── Stage 1 ───────────────────────────────────────────────────────────────
    def _load(self) -> None:
        df = pd.read_csv(self.filepath, parse_dates=['Date'], index_col='Date')
        df.columns = df.columns.str.strip()
        # Remove duplicate dates, keep the most recent entry
        self.raw = df.astype(float).loc[~df.index.duplicated(keep='last')]

    def _repair_cross_section(self) -> None:
        """Spline-interpolate zeros and NaNs across the tenor dimension."""
        df = self.raw.copy().replace(0.0, np.nan)
        if df.isna().sum().sum() == 0:
            self.raw = df
            return
        original_cols = df.columns.tolist()
        # Temporarily label columns with numeric tenors for interpolation axis
        df.columns = [0.25, 0.5, 0.75, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0]
        df.interpolate(method='cubicspline', axis=1, inplace=True)
        df.columns = original_cols
        self.raw = df

    # ── Stage 2 & 3 ───────────────────────────────────────────────────────────
    def _align_and_floor(self) -> None:
        self.clean = (self.raw
                      .asfreq('B')          # business-day calendar
                      .ffill()              # forward-fill gaps only
                      .dropna()
                      .clip(lower=0.0001))  # 1 bp floor

    # ── Stage 4 ───────────────────────────────────────────────────────────────
    def _mad_filter(self, window: int = 20, k: float = 5.0) -> None:
        df   = self.clean.copy()
        rmed = df.rolling(window, min_periods=1).median()
        adev = (df - rmed).abs()
        # Offset by 1 bp to avoid division by zero in flat-rate periods
        rmad = adev.rolling(window, min_periods=1).median() + 0.0001
        self.clean = df.mask(adev > k * rmad, rmed)

    def run(self) -> pd.DataFrame:
        self._load()
        self._repair_cross_section()
        self._align_and_floor()
        self._mad_filter()
        print(f'[YieldPipeline] {self.filepath.split("/")[-1]} → shape {self.clean.shape}')
        print(f'  Date range : {self.clean.index[0].date()} → {self.clean.index[-1].date()}')
        print(f'  NaN remaining : {self.clean.isna().sum().sum()}')
        return self.clean

    def plot(self, title: str = '') -> None:
        if self.clean is None:
            return
        fig, ax = plt.subplots(figsize=(13, 4))
        for col, lbl in zip(['ZC025YR','ZC200YR','ZC3000YR'],
                             ['3-Month','2-Year','30-Year']):
            if col in self.clean.columns:
                ax.plot(self.clean.index, self.clean[col] * 100,
                        lw=0.9, label=lbl)
        ax.set(title=title or 'Cleaned Yield Series',
               xlabel='Date', ylabel='Yield (%)')
        ax.legend()
        plt.tight_layout(); plt.show()

In [ ]:
print('=' * 55)
print('TRAINING DATA')
print('=' * 55)
train_pipe = YieldPipeline(TRAIN_CSV)
train_df   = train_pipe.run()
train_pipe.plot('Cleaned Training Yields — Short / Belly / Long End')
display(train_df.describe().map(lambda x: f'{x*100:.4f}%'))

## 2. Base CIR Model — Affine Term Structure & Cross-Sectional Calibration
### 2.1 Bond Pricing via Affine Term Structure
The CIR model is a member of the **Affine Term Structure Model (ATSM)** class. Zero-coupon bond prices take the exponentially affine form:
$$P(t,T) = A(\tau)\,e^{-B(\tau)\,r_t}, \quad \tau = T - t$$

Rearranging into continuously compounded yield space:
$$R(t,\tau) = \frac{B(\tau)\,r_t - \ln A(\tau)}{\tau}$$

The deterministic loading functions $A(\tau)$ and $B(\tau)$ solve a pair of Riccati ODEs. Setting $h = \sqrt{\kappa^2 + 2\sigma^2}$:
$$B(\tau) = \frac{2(e^{h\tau}-1)}{2h + (\kappa+h)(e^{h\tau}-1)}$$
$$\ln A(\tau) = \frac{2\kappa\theta}{\sigma^2}\ln\!\left[\frac{2h\,e^{(\kappa+h)\tau/2}}{2h+(\kappa+h)(e^{h\tau}-1)}\right]$$

### 2.2 Calibration Objective — Tenor-Weighted MSE with Volatility Anchor
Standard gradient solvers (BFGS, Nelder-Mead) fail on CIR loss surfaces due to multiple local minima. This implementation uses **Differential Evolution**, a population-based global algorithm.

The objective minimised is a **Tenor-Weighted MSE** plus a regularisation penalty on $\sigma$:
$$\mathcal{L}(\kappa,\theta,\sigma) = \frac{1}{N}\sum_{i}\left[(R_{\text{obs}}(\tau_i)-R_{\text{CIR}}(\tau_i))\cdot\sqrt{\tau_i}\right]^2 + \lambda\cdot\text{Penalty}(\sigma)$$

The $\sqrt{\tau_i}$ weights prevent the optimiser from sacrificing long-end accuracy, which anchors $\theta$. The penalty term compares $\sigma$ against the **empirical variance proxy**:
$$\hat{\sigma}^2 \approx \mathbb{E}\!\left[\frac{(\Delta r)^2}{r_t\,\Delta t}\right]$$
and fires only when the candidate $\sigma$ deviates more than 50% from $\hat{\sigma}$.

### 2.3 Feller Condition Enforcement
Any candidate parameter triplet violating $2\kappa\theta \leq \sigma^2$ is immediately rejected by returning a loss of $10^{10}$, hard-wiring the square-root process to remain strictly positive.

In [ ]:
class CIRModel:
    """
    Single-factor CIR model calibrated via cross-sectional Differential Evolution.
    Mathematical core
    -----------------
    SDE  : dr = κ(θ - r)dt + σ√r dW
    Yield: R(τ) = [B(τ)·r - lnA(τ)] / τ
    where h = √(κ² + 2σ²) and the Riccati solutions are
        B(τ) = 2(e^{hτ}-1) / [2h + (κ+h)(e^{hτ}-1)]
        lnA(τ) = (2κθ/σ²) · ln[2h·exp((κ+h)τ/2) / denominator]
    """
    SHORT_RATE_COL   = 'ZC025YR'
    TARGET_COLS      = ['ZC050YR','ZC075YR','ZC100YR','ZC200YR',
                        'ZC500YR','ZC1000YR','ZC2000YR','ZC3000YR']
    TARGET_TENORS    = np.array([0.5, 0.75, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0])

    def __init__(self, train_df: pd.DataFrame):
        print('[CIRModel] Initialising cross-sectional environment...')
        self._r_proxy  = train_df[self.SHORT_RATE_COL].values
        self._Y_market = train_df[self.TARGET_COLS].values
        self._dt       = 1.0 / 252.0
        # Pre-compute empirical σ once — used as anchor in calibration bounds
        dr   = np.diff(self._r_proxy)
        r_m  = np.maximum(self._r_proxy[:-1], 1e-8)
        self._emp_sigma = float(np.sqrt(np.mean(dr**2 / (r_m * self._dt))))
        print(f'  Empirical σ̂ = {self._emp_sigma:.6f}')
        self.kappa = self.theta = self.sigma = None
        self._calibrated = False

    # ── Affine factors ────────────────────────────────────────────────────────
    @staticmethod
    def _AB(kappa, theta, sigma, tau):
        """Vectorised computation of B(τ) and lnA(τ) for an array of maturities."""
        h     = np.sqrt(kappa**2 + 2.0 * sigma**2)
        eht   = np.exp(h * tau)
        denom = 2.0 * h + (kappa + h) * (eht - 1.0)
        B     = 2.0 * (eht - 1.0) / denom
        lnA   = (2.0 * kappa * theta / sigma**2) * np.log(
                    2.0 * h * np.exp((kappa + h) * tau / 2.0) / denom)
        return lnA, B

    def yield_curve(self, r_t: float, tau: np.ndarray) -> np.ndarray:
        """Theoretical CIR yields given short rate r_t and maturity array tau."""
        if not self._calibrated:
            raise RuntimeError('Call .calibrate() first.')
        lnA, B = self._AB(self.kappa, self.theta, self.sigma, np.asarray(tau))
        return (B * r_t - lnA) / np.asarray(tau)

    # ── Loss function ─────────────────────────────────────────────────────────
    def _loss(self, params):
        kappa, theta, sigma = params
        # Hard-reject Feller violations and non-positive parameters
        if kappa <= 0 or theta <= 0 or sigma <= 0:
            return 1e10
        if 2.0 * kappa * theta <= sigma**2:
            return 1e10
        lnA, B = self._AB(kappa, theta, sigma, self.TARGET_TENORS)
        Y_hat  = (self._r_proxy[:, None] * B - lnA) / self.TARGET_TENORS
        # Tenor-weighted squared errors — upweights long end to pin θ
        w      = np.sqrt(self.TARGET_TENORS)
        xs_mse = np.mean(((self._Y_market - Y_hat) * w) ** 2)
        # σ anchor penalty — fires beyond 50% deviation from empirical σ
        dev     = abs(sigma - self._emp_sigma) / self._emp_sigma
        penalty = max(0.0, dev - 0.5) ** 2
        return xs_mse + 0.1 * penalty

    # ── Calibration ───────────────────────────────────────────────────────────
    def calibrate(self, popsize: int = 15, tol: float = 1e-6, seed: int = 42):
        if self._calibrated:
            print('[CIRModel] Already calibrated — skipping.')
            return
        bounds = [
            (1e-4, 5.0),
            (1e-4, 0.20),
            (self._emp_sigma * 0.3, self._emp_sigma * 3.0),
        ]
        print('[CIRModel] Launching Differential Evolution...')
        res = differential_evolution(
            self._loss, bounds,
            strategy='best1bin', popsize=popsize,
            recombination=0.7, tol=tol, seed=seed,
        )
        if res.success:
            self.kappa, self.theta, self.sigma = res.x
            self._calibrated = True
            feller = 2 * self.kappa * self.theta - self.sigma**2
            print('\n  Calibration succeeded')
            print(f'  κ = {self.kappa:.6f}  |  θ = {self.theta*100:.4f}%'
                  f'  |  σ = {self.sigma:.6f}')
            print(f'  Feller slack 2κθ − σ² = {feller:.6f}  ✓')
        else:
            print(f'  Calibration failed: {res.message}')

    # ── Prediction ────────────────────────────────────────────────────────────
    def predict_panel(self, df: pd.DataFrame) -> np.ndarray:
        """Return (T × n_tenors) predicted yield matrix for a given DataFrame."""
        r_vals = df[self.SHORT_RATE_COL].values
        tau    = self.TARGET_TENORS
        lnA, B = self._AB(self.kappa, self.theta, self.sigma, tau)
        return (r_vals[:, None] * B - lnA) / tau

In [ ]:
cir = CIRModel(train_df)
cir.calibrate()

## 2.4 In-Sample Fit: Single-Day Visual Check

In [ ]:
# ── Single-day snapshot ───────────────────────────────────────────────────────
snap_date   = train_df.index[-1]
snap_row    = train_df.loc[snap_date]
r_snap      = snap_row['ZC025YR']
tau_plot    = CIRModel.TARGET_TENORS
col_plot    = CIRModel.TARGET_COLS

Y_actual    = snap_row[col_plot].values
Y_predicted = cir.yield_curve(r_snap, tau_plot)

fig, ax = plt.subplots(figsize=(9, 5))
full_tau = np.concatenate([[0.25], tau_plot])
ax.plot(full_tau, np.concatenate([[r_snap], Y_actual])  * 100,
        'o-', color=PALETTE['actual'],    lw=2, ms=7, label='Market')
ax.plot(full_tau, np.concatenate([[r_snap], Y_predicted])* 100,
        's--', color=PALETTE['predicted'], lw=2, ms=6, label='CIR')

ax.set(title=f'In-Sample Fit — {snap_date.date()}',
       xlabel='Maturity (yr)', ylabel='Yield (%)')
ax.legend(); plt.tight_layout(); plt.show()

mse_snap = float(np.mean((Y_actual - Y_predicted)**2))
print(f'Single-day MSE : {mse_snap:.8f}')

In [ ]:
# ── Full in-sample panel backtest ────────────────────────────────────────────
print('Running full in-sample backtest...')
Y_hat_train  = cir.predict_panel(train_df)
Y_true_train = train_df[CIRModel.TARGET_COLS].values

flat_true = Y_true_train.flatten()
flat_pred = Y_hat_train.flatten()
mse_is  = float(np.mean((flat_true - flat_pred)**2))
rmse_is = float(np.sqrt(mse_is))
r2_is   = float(1 - np.sum((flat_true - flat_pred)**2)
                      / np.sum((flat_true - flat_true.mean())**2))

print(f'\n  In-Sample MSE  : {mse_is:.8f}')
print(f'  In-Sample RMSE : {rmse_is*100:.4f}%  ({rmse_is*1e4:.2f} bps)')
print(f'  In-Sample R²   : {r2_is:.5f}')

## 3. Out-of-Sample Backtest — Base CIR
### 3.1 Forward-Test Protocol
Strict data isolation is the foundation of a credible backtest. The test set is processed through the identical `YieldPipeline` as the training set, and the calibrated parameters ($\kappa, \theta, \sigma$) are **completely frozen** before any test observation is touched.

For each test-set day, the model receives only the **3-Month yield** as input and generates blind predictions for all other tenors. No test-set yield beyond 3M enters the model at any point.

### 3.2 Evaluation Metrics
| Metric | Interpretation |
|--------|---------------|
| $MSE = \frac{1}{N}\sum(y_i-\hat y_i)^2$ | Average squared basis-point error |
| $RMSE = \sqrt{MSE}$ | Error in same units as yield; interpretable as bps deviation |
| $R^2 = 1 - SS_{res}/SS_{tot}$ | Fraction of yield variance captured by the model |

### 3.3 Empirical Results
Frozen CIR parameters tested against **523 days** of unseen market data:
- Target tenors: 6M, 9M, 1Y, 2Y (tenors present in the test set)
- Out-of-Sample $R^2 = 0.8755$ — strong evidence of structural parameter stability
- RMSE ≈ 24 bps — tight fit considering only a single 3M input drives predictions

The remaining ~12% unexplained variance is attributable to the single-factor perfect-correlation constraint: the base CIR curve is monotonic and cannot reproduce slope twists or belly humps. This motivates the three extensions in Sections 4–6.

In [ ]:
# ── Load & clean test data ────────────────────────────────────────────────────
print('=' * 55)
print('TEST DATA')
print('=' * 55)
test_pipe = YieldPipeline(TEST_CSV)
test_df   = test_pipe.run()

# Discover which target columns are present in the test set
test_target_cols   = [c for c in CIRModel.TARGET_COLS if c in test_df.columns]
test_target_tenors = np.array([COL_TO_TENOR[c] for c in test_target_cols])

print(f'\nTest tenors active: {[f"{t}Y" for t in test_target_tenors]}')
print(f'Running blind prediction on {len(test_df)} test days...')

r_test    = test_df['ZC025YR'].values
lnA_t, B_t = CIRModel._AB(cir.kappa, cir.theta, cir.sigma, test_target_tenors)
Y_hat_test = (r_test[:, None] * B_t - lnA_t) / test_target_tenors
Y_true_test = test_df[test_target_cols].values

flat_true_t = Y_true_test.flatten()
flat_pred_t = Y_hat_test.flatten()
mse_oos  = float(np.mean((flat_true_t - flat_pred_t)**2))
rmse_oos = float(np.sqrt(mse_oos))
r2_oos   = float(1 - np.sum((flat_true_t - flat_pred_t)**2)
                       / np.sum((flat_true_t - flat_true_t.mean())**2))

print(f'\n  OOS MSE  : {mse_oos:.8f}')
print(f'  OOS RMSE : {rmse_oos*100:.4f}%  ({rmse_oos*1e4:.2f} bps)')
print(f'  OOS R²   : {r2_oos:.5f}  ',
      '✓ Target met' if r2_oos > 0.85 else '✗ Below 0.85 target')

### 3.4 Visual Validation — Predicted vs Actual Yield Curves
Four equidistant days are sampled across the 523-day test window. The **solid line** is the actual market curve; the **dashed line** is the model's out-of-sample prediction anchored solely on the 3-Month rate. A tight dashed/solid overlap across all maturities confirms that the cross-sectionally calibrated parameters generalize well across time.

In [ ]:
n_days   = len(test_df)
snap_idx = [0, n_days // 3, 2 * n_days // 3, n_days - 1]
snap_dates = test_df.index[snap_idx]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('OOS Yield Curve — Base CIR (3M input only)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for ax, date, si in zip(axes, snap_dates, snap_idx):
    row     = test_df.loc[date]
    r_in    = row['ZC025YR']
    Y_act   = row[test_target_cols].values
    Y_pred  = Y_hat_test[si]

    full_tau = np.concatenate([[0.25], test_target_tenors])
    ax.plot(full_tau, np.concatenate([[r_in], Y_act])  * 100,
            'o-', color=PALETTE['actual'],    lw=2, ms=6, label='Actual')
    ax.plot(full_tau, np.concatenate([[r_in], Y_pred]) * 100,
            's--', color=PALETTE['predicted'], lw=2, ms=5, label='CIR')

    ax.set(title=str(date.date()), xlabel='Maturity (yr)', ylabel='Yield (%)')
    ax.legend(fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

## 4. Extension I — Affine Jump-Diffusion (Duffie-Pan-Singleton 2000)
### 4.1 Motivation
The standardised residuals from CIR calibration exhibit **fat tails and positive skew** — the empirical signature of discrete, sudden jumps in the short rate driven by central bank policy surprises, banking shocks, or geopolitical events. Pure Brownian diffusion cannot generate these discontinuities.

### 4.2 The AJD Short-Rate SDE
The jump-diffusion extension augments the CIR SDE with a compound Poisson process $J_t$:
$$dr_t = \kappa(\theta - r_t)\,dt + \sigma\sqrt{r_t}\,dW_t + dJ_t$$
Jump arrivals follow a **Poisson process** with intensity $\lambda$ (jumps per year). Jump magnitudes are drawn from an **Exponential distribution** with mean $\mu_J$ — ensuring all jumps are strictly positive, preserving the non-negative rate constraint.

### 4.3 Quasi-Closed-Form Bond Pricing
The AJD model remains within the affine class. The $B(\tau)$ loading function is unchanged from base CIR. The log-A factor picks up a jump premium:
$$\ln A_{\text{AJD}}(\tau) = \ln A_{\text{CIR}}(\tau) - \lambda\int_0^\tau \left(1 - \frac{1}{1+\mu_J B(u)}\right)du$$
The integral is evaluated via `scipy.integrate.quad` for each maturity.

### 4.4 Kalman Filter for Latent State Extraction
In the out-of-sample window, $r_t$ is unobservable. An **Extended Kalman Filter** infers the latent short rate by processing the full yield curve simultaneously.

**Prediction step** (Euler SDE + jump drift):
$$r_{t|t-1} = r_{t-1} + \left[\kappa(\theta - r_{t-1}) + \lambda\mu_J\right]\Delta t$$
$$P_{t|t-1} = (1 - \kappa\Delta t)^2 P_{t-1} + \sigma^2 r_{t-1}\Delta t + \lambda(2\mu_J^2)\Delta t$$

**Update step** (Kalman gain):
$$K_t = P_{t|t-1}H^T\left(HP_{t|t-1}H^T + R\right)^{-1}$$
$$r_{t|t} = r_{t|t-1} + K_t\left(Z_t - Hr_{t|t-1} - d\right)$$

In [ ]:
class AffineJumpDiffusion:
    """
    CIR model extended with a compound Poisson jump process.
    Reference: Duffie, Pan & Singleton (2000).

    Parameters
    ----------
    k_rev      : κ  — mean-reversion speed
    theta_mean : θ  — long-run mean rate
    vol        : σ  — diffusion volatility
    j_freq     : λ  — Poisson jump intensity (jumps / year)
    j_size     : μJ — mean exponential jump size
    """
    def __init__(self):
        self.k_rev = self.theta_mean = self.vol = None
        self.j_freq = self.j_size = None
        self._fitted = False

    # ── Riccati factors ───────────────────────────────────────────────────────
    @staticmethod
    def _cir_AB(k, th, s, tau):
        g      = np.sqrt(k**2 + 2.0 * s**2)
        egtau  = np.exp(g * tau)
        denom  = (g + k) * (egtau - 1.0) + 2.0 * g
        B      = 2.0 * (egtau - 1.0) / denom
        lnA    = (2.0 * k * th / s**2) * np.log(
                     2.0 * g * np.exp((k + g) * tau / 2.0) / denom)
        return lnA, B

    def _ajd_factors(self, k, th, s, lam, mu_j, tau_arr):
        lnA_base, B_base = self._cir_AB(k, th, s, tau_arr)
        lnA_jump = np.zeros_like(lnA_base)
        for i, tau in enumerate(tau_arr):
            if tau == 0.0:
                continue
            g = np.sqrt(k**2 + 2.0 * s**2)
            def integrand(u):
                egu   = np.exp(g * u)
                den_u = (g + k) * (egu - 1.0) + 2.0 * g
                Bu    = 2.0 * (egu - 1.0) / den_u
                return lam * (1.0 - 1.0 / (1.0 + mu_j * Bu))
            adj, _ = integrate.quad(integrand, 0.0, tau)
            lnA_jump[i] = -adj
        return lnA_base + lnA_jump, B_base

    # ── Calibration ───────────────────────────────────────────────────────────
    def fit(self, r_proxy, Y_market, maturities):
        def objective(p):
            k, th, s, lf, ls = p
            lnA, B = self._ajd_factors(k, th, s, lf, ls, maturities)
            Y_hat  = (r_proxy[:, None] * B - lnA) / maturities
            return float(np.mean((Y_market - Y_hat)**2))

        bounds = [(1e-3,5.0),(1e-3,1.0),(1e-3,1.0),(0.0,10.0),(1e-4,0.1)]
        feller = NonlinearConstraint(lambda p: 2*p[0]*p[1] - p[2]**2, 0, np.inf)

        print('[AJD] Running Differential Evolution (constrained)...')
        res = differential_evolution(objective, bounds,
                                     constraints=(feller,),
                                     strategy='best1bin', maxiter=1000,
                                     popsize=15, tol=1e-6, seed=42)

        self.k_rev, self.theta_mean, self.vol, self.j_freq, self.j_size = res.x
        self._fitted = True
        print(f'  κ={self.k_rev:.4f}  θ={self.theta_mean*100:.3f}%'
              f'  σ={self.vol:.4f}  λ={self.j_freq:.3f}  μJ={self.j_size*1e4:.1f}bps')
        return res.success

    # ── Kalman filter ─────────────────────────────────────────────────────────
    def filter_states(self, Y_obs, maturities, dt=1/252, R_noise=1e-5):
        """Extended Kalman Filter to extract latent r_t from the yield panel."""
        if not self._fitted:
            raise RuntimeError('Call .fit() first.')

        T = len(Y_obs)
        states = np.zeros(T)

        lnA, B = self._ajd_factors(
            self.k_rev, self.theta_mean, self.vol,
            self.j_freq, self.j_size, maturities)

        H = B / maturities
        d = -lnA / maturities
        R_mat = np.eye(len(maturities)) * R_noise

        r_hat = self.theta_mean
        P     = self.vol**2 * self.theta_mean / (2.0 * self.k_rev)

        for i in range(T):
            # Predict
            r_prior = (r_hat + (self.k_rev * (self.theta_mean - r_hat)
                        + self.j_freq * self.j_size) * dt)
            j_var   = self.j_freq * 2.0 * self.j_size**2 * dt
            P_prior = ((1 - self.k_rev * dt)**2 * P
                       + self.vol**2 * max(r_hat, 1e-6) * dt + j_var)

            # Update
            innov  = Y_obs[i] - (d + H * r_prior)
            S      = np.outer(H, H) * P_prior + R_mat
            K      = P_prior * H @ np.linalg.inv(S)
            r_hat  = r_prior + float(np.dot(K, innov))
            P      = (1 - float(np.dot(K, H))) * P_prior
            states[i] = r_hat

        return states

In [ ]:
# ── Training tensors ─────────────────────────────────────────────────────────
train_taus  = np.array([0.5, 0.75, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0])
train_cols  = ['ZC050YR','ZC075YR','ZC100YR','ZC200YR',
               'ZC500YR','ZC1000YR','ZC2000YR','ZC3000YR']
r_tr        = train_df['ZC025YR'].values
Y_tr        = train_df[train_cols].values

# ── Test tensors ──────────────────────────────────────────────────────────────
test_cols_ajd  = [c for c in train_cols if c in test_df.columns]
test_taus_ajd  = np.array([COL_TO_TENOR[c] for c in test_cols_ajd])
Y_te_ajd       = test_df[test_cols_ajd].values

# ── Fit ───────────────────────────────────────────────────────────────────────
ajd = AffineJumpDiffusion()
ajd.fit(r_proxy=r_tr, Y_market=Y_tr, maturities=train_taus)

# ── Kalman filter on test ──────────────────────────────────────────────────────
print('\n[AJD] Running Kalman filter on test set...')
r_filtered_ajd = ajd.filter_states(Y_te_ajd, test_taus_ajd)
lnA_te, B_te = ajd._ajd_factors(ajd.k_rev, ajd.theta_mean, ajd.vol,
                                  ajd.j_freq, ajd.j_size, test_taus_ajd)
Y_hat_ajd = (r_filtered_ajd[:, None] * B_te - lnA_te) / test_taus_ajd

flat_a = Y_te_ajd.flatten();  flat_p = Y_hat_ajd.flatten()
r2_ajd = float(r2_score(flat_a, flat_p))
print(f'\n  AJD OOS R² = {r2_ajd:.5f}',
      '  ✓' if r2_ajd > 0.85 else '  ✗')

### 4.5 Yield Curve Snapshots — AJD Model
Four test-set dates are visualised. The latent rate $r_t$ is extracted at each date by the Kalman filter using the full observed yield curve, then projected through the AJD affine factors to generate the full theoretical curve.

In [ ]:
n_days_t    = len(test_df)
snap_idx_t  = [0, n_days_t // 3, 2 * n_days_t // 3, n_days_t - 1]
snap_dates_t = test_df.index[snap_idx_t]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('OOS Yield Curve — AJD Model (Kalman state)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for ax, date, si in zip(axes, snap_dates_t, snap_idx_t):
    row    = test_df.loc[date]
    r_in   = row['ZC025YR']
    Y_act  = row[test_cols_ajd].values
    Y_pred = Y_hat_ajd[si]

    full_tau = np.concatenate([[0.25], test_taus_ajd])
    ax.plot(full_tau, np.concatenate([[r_in], Y_act])  * 100,
            'o-', color=PALETTE['actual'],    lw=2, ms=6, label='Market')
    ax.plot(full_tau, np.concatenate([[r_in], Y_pred]) * 100,
            '^--', color=PALETTE['predicted'], lw=2, ms=5, label='AJD')

    ax.set(title=str(date.date()), xlabel='Maturity (yr)', ylabel='Yield (%)')
    ax.legend(fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

## 5. Extension II — CIR++ Deterministic Shift (Brigo-Mercurio 2001)
### 5.1 The Perfect-Fit Extension
The base CIR model constrains the entire yield curve to a fixed shape governed by $\kappa, \theta, \sigma$. This makes it structurally impossible to match an arbitrary initial term structure. **Brigo and Mercurio (2001)** introduced a deterministic shift $\varphi(t)$ that restores perfect initial fit while preserving the CIR analytical tractability:
$$r_t = x_t + \varphi(t)$$
where $x_t$ follows the standard CIR SDE. The resulting yield formula is:
$$R^{CIR++}(t,\tau) = R^{CIR}(x_t,\tau) + \Psi(\tau)$$

### 5.2 Two-Stage Calibration
**Stage 1:** Global Differential Evolution minimises a Tenor-Weighted MSE to find the optimal $(\kappa, \theta, \sigma)$ for the structural dynamics of $x_t$. The sigma penalty prevents $\sigma$ collapsing to zero.
**Stage 2:** The shift function $\Psi(\tau)$ is computed as the **sample-average residual** between market yields and base CIR yields across the training set:
$$\Psi(\tau) = \frac{1}{N}\sum_{i=1}^{N}\left[R_{\text{market},i}(\tau) - R^{CIR}(x_{i},\tau)\right]$$
Averaging across $\sim$99 training dates prevents overfitting to any single day's noise. A **Cubic Spline** makes $\Psi$ continuous and differentiable across arbitrary maturities.

### 5.3 Kalman Filter State Extraction
During the backtest, the latent $x_t$ is extracted by a Kalman filter that integrates the CIR++ observation equation, with the shift correction absorbed into the intercept vector $d$:
$$Z_t = H x_t + d + v, \quad H = \frac{B(\tau)}{\tau}, \quad d = \frac{-\ln A(\tau)}{\tau} + \Psi(\tau)$$

In [ ]:
class CIRPlusPlus:
    """
    CIR++ model (Brigo-Mercurio deterministic shift + Kalman filter).

    Architecture
    ------------
    1. Calibrate base CIR parameters (κ, θ, σ) via Differential Evolution.
    2. Build the average yield-residual spline φ(τ) across the training set.
    3. Extract latent x_t in the test window via Extended Kalman Filter.
    4. Predict full curve as CIR_yield(x_t, τ) + φ(τ).
    """
    SHORT_COL    = 'ZC025YR'
    TARGET_COLS  = ['ZC050YR','ZC075YR','ZC100YR','ZC200YR',
                    'ZC500YR','ZC1000YR','ZC2000YR','ZC3000YR']
    TARGET_TAUS  = np.array([0.5, 0.75, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0])

    def __init__(self, train_df: pd.DataFrame):
        print('[CIR++] Initialising environment...')
        self._r  = train_df[self.SHORT_COL].values
        self._Y  = train_df[self.TARGET_COLS].values
        self._dt = 1.0 / 252.0

        dr = np.diff(self._r)
        rm = np.maximum(self._r[:-1], 1e-8)
        self._emp_sig = float(np.sqrt(np.mean(dr**2 / (rm * self._dt))))
        print(f'  σ̂_empirical = {self._emp_sig:.6f}')
        self.kappa = self.theta = self.sigma = None
        self._phi  = None
        self._ready = False

    # ── Affine factors (shared) ───────────────────────────────────────────────
    @staticmethod
    def _AB(k, th, s, tau):
        h    = np.sqrt(k**2 + 2.0*s**2)
        eht  = np.exp(h * tau)
        den  = 2.0*h + (k+h)*(eht - 1.0)
        B    = 2.0*(eht - 1.0) / den
        lnA  = (2.0*k*th / s**2) * np.log(2.0*h*np.exp((k+h)*tau/2.0) / den)
        return lnA, B

    def _cir_yield(self, k, th, s, r0, tau):
        lnA, B = self._AB(k, th, s, tau)
        return (B * r0 - lnA) / tau

    # ── Stage-1 loss ─────────────────────────────────────────────────────────
    def _loss(self, p):
        k, th, s = p
        if k<=0 or th<=0 or s<=0 or 2*k*th<=s**2:
            return 1e10

        lnA, B = self._AB(k, th, s, self.TARGET_TAUS)
        Y_hat  = (self._r[:,None]*B - lnA) / self.TARGET_TAUS
        w      = np.where(self.TARGET_TAUS <= 2.0, 3.0, 1.0)
        xs_mse = float(np.mean(((self._Y - Y_hat)*w)**2))
        dev    = abs(s - self._emp_sig) / self._emp_sig
        return xs_mse + 0.01 * max(0, dev - 0.5)**2

    # ── Stage-2 shift spline ──────────────────────────────────────────────────
    def _build_phi(self):
        tau_fine = np.linspace(0.25, 30.0, 300)
        residuals = []
        for i in range(0, len(self._r), 21):       # sample every ~21 trading days
            y_cir = self._cir_yield(self.kappa, self.theta, self.sigma,
                                    self._r[i], self.TARGET_TAUS)
            phi_at_knots = self._Y[i] - y_cir
            tau_knots = np.concatenate([[0.25], self.TARGET_TAUS])
            phi_knots = np.concatenate([[0.0],  phi_at_knots])

            sp = CubicSpline(tau_knots, phi_knots, bc_type='not-a-knot')
            residuals.append(sp(tau_fine))

        phi_avg = np.mean(residuals, axis=0)
        self._phi = CubicSpline(tau_fine, phi_avg)
        n = len(residuals)
        print(f'  φ(τ) built from {n} training samples. '
              f'Range: [{phi_avg.min():.5f}, {phi_avg.max():.5f}]')

    # ── Calibrate ─────────────────────────────────────────────────────────────
    def calibrate(self):
        bounds = [(1e-4,5.0),(1e-4,0.20),
                  (self._emp_sig*0.3, self._emp_sig*3.0)]

        print('[CIR++] Stage 1 — Differential Evolution...')
        res = differential_evolution(self._loss, bounds,
                                     strategy='best1bin', popsize=15,
                                     recombination=0.7, mutation=(0.5,1.0),
                                     tol=1e-7, seed=42, polish=True)
        self.kappa, self.theta, self.sigma = res.x
        print(f'  κ={self.kappa:.6f}  θ={self.theta*100:.4f}%  σ={self.sigma:.6f}')

        print('[CIR++] Stage 2 — Building φ(τ) spline...')
        self._build_phi()
        self._ready = True
        print('[CIR++] Ready.')

    # ── Kalman filter ─────────────────────────────────────────────────────────
    def filter_states(self, Y_obs, tau_obs, R_noise=1e-5):
        lnA, B  = self._AB(self.kappa, self.theta, self.sigma, tau_obs)
        phi_obs = self._phi(tau_obs)

        H = B / tau_obs
        d = -lnA / tau_obs + phi_obs
        R = np.eye(len(tau_obs)) * R_noise

        x_hat = self.theta
        P     = self.sigma**2 * self.theta / (2.0 * self.kappa)
        states = np.zeros(len(Y_obs))

        for i in range(len(Y_obs)):
            x_p = max(x_hat + self.kappa*(self.theta - x_hat)*self._dt, 1e-6)
            P_p = (1 - self.kappa*self._dt)**2 * P + self.sigma**2 * x_p * self._dt

            y   = Y_obs[i] - (d + H*x_p)
            S   = np.outer(H,H)*P_p + R
            K   = P_p * H @ np.linalg.inv(S)
            x_hat = max(x_p + float(np.dot(K,y)), 1e-6)
            P     = (1 - float(np.dot(K,H))) * P_p
            states[i] = x_hat

        return states

    # ── Predict ───────────────────────────────────────────────────────────────
    def predict(self, x_t, tau):
        tau = np.asarray(tau)
        lnA, B = self._AB(self.kappa, self.theta, self.sigma, tau)
        return (B*x_t - lnA)/tau + self._phi(tau)

    # ── Backtest ──────────────────────────────────────────────────────────────
    def backtest(self, test_df):
        avail      = {t: c for t, c in zip(self.TARGET_TAUS, self.TARGET_COLS)
                      if c in test_df.columns}
        tau_bt     = np.array(sorted(avail))
        col_bt     = [avail[t] for t in tau_bt]
        Y_obs      = test_df[col_bt].values

        print(f'[CIR++] Kalman filter on {len(test_df)} test days...')
        x_seq = self.filter_states(Y_obs, tau_bt)
        Y_hat = np.stack([self.predict(x, tau_bt) for x in x_seq])

        r2_per = {}
        for i, tau in enumerate(tau_bt):
            yt, yp = Y_obs[:,i], Y_hat[:,i]
            r2_per[tau] = float(1 - np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2))

        overall = float(np.mean(list(r2_per.values())))
        print('\n  OOS R² by tenor:')
        print('  ' + '─'*40)
        for t, r2 in r2_per.items():
            bar = '█' * int(max(0,r2)*20)
            print(f'  {t:5.2f}yr │ {bar:<20} │ {r2:.4f}')

        print(f'\n  Overall R² = {overall:.4f}')
        return r2_per, overall, Y_hat, tau_bt

In [ ]:
cir_pp = CIRPlusPlus(train_df)
cir_pp.calibrate()

# Verify spline at key tenors
check_taus = np.array([0.5, 1.0, 2.0, 5.0, 10.0])
print('\nφ(τ) spot check (bps):')
for t, v in zip(check_taus, cir_pp._phi(check_taus)):
    print(f'  τ={t:.2f}yr  →  {v*1e4:+.2f} bps')

r2_pp, overall_r2_pp, preds_pp, taus_pp = cir_pp.backtest(test_df)

### 5.4 Yield Curve Snapshots — CIR++ Model
The spline-corrected CIR++ curve is drawn in blue. The deterministic shift $\Psi(\tau)$ is visible as the vertical distance between the CIR++ line and what a pure CIR curve would predict — it systematically lifts or depresses specific maturity buckets to match the historical average shape of market residuals.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('OOS Yield Curve — CIR++ (Brigo-Mercurio Shift + Kalman)',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

col_pp = [c for c in CIRPlusPlus.TARGET_COLS if c in test_df.columns]
for ax, date, si in zip(axes, snap_dates_t, snap_idx_t):
    row   = test_df.loc[date]
    r_in  = row['ZC025YR']
    Y_act = row[col_pp].values
    Y_pr  = preds_pp[si]

    full_tau = np.concatenate([[0.25], taus_pp])
    ax.plot(full_tau, np.concatenate([[r_in], Y_act]) * 100,
            'o-',  color=PALETTE['actual'],    lw=2, ms=6, label='Market')
    ax.plot(full_tau, np.concatenate([[r_in], Y_pr])  * 100,
            's--', color='#457b9d',             lw=2, ms=5, label='CIR++')

    ax.set(title=str(date.date()), xlabel='Maturity (yr)', ylabel='Yield (%)')
    ax.legend(fontsize=8)

plt.tight_layout(rect=[0,0,1,0.95]); plt.show()

## 6. Extension III — Two-Factor CIR with Algebraic State Extraction
### 6.1 Why Two Factors?
A single-factor model imposes **perfect correlation** across the entire yield curve: every tenor moves in the same direction by an amount proportional to its $B(\tau)$ loading. Real curves exhibit twists (slope changes) and humps (curvature changes) driven by the differential sensitivity of short vs. long rates to different economic forces.

The Two-Factor CIR decomposes the short rate into **two independent latent drivers**:
$$r_t = x_{1,t} + x_{2,t}$$
Each factor satisfies its own square-root mean-reverting SDE:
$$dx_{k,t} = \kappa_k(\theta_k - x_{k,t})\,dt + \sigma_k\sqrt{x_{k,t}}\,dW_{k,t}, \quad k=1,2$$
Because $dW_1 \perp dW_2$, the yield pricing formula sums additively:
$$R(t,\tau) = \frac{B_1(\tau)x_{1,t} + B_2(\tau)x_{2,t} - \ln A_1(\tau) - \ln A_2(\tau)}{\tau}$$

### 6.2 Algebraic Anchor Inversion
Rather than tracking $x_1, x_2$ through a Kalman Filter, this implementation uses an **exact algebraic inversion**. Pinning two observable yields at anchor maturities $\tau_1, \tau_2$ produces a $2\times2$ linear system:
$$\begin{bmatrix}B_1(\tau_1) & B_2(\tau_1) \\ B_1(\tau_2) & B_2(\tau_2)\end{bmatrix}\begin{bmatrix}x_1 \\ x_2\end{bmatrix} = \begin{bmatrix}Z_1 \\ Z_2\end{bmatrix}$$
where $Z_k = R_{\text{mkt}}(\tau_k)\cdot\tau_k + \ln A_1(\tau_k) + \ln A_2(\tau_k)$.
Solving via Cramer's rule gives instantaneous, exact state extraction at every time step with no filter tuning.

### 6.3 Global-Local Hybrid Calibration
Six parameters must be jointly optimised. A **two-phase strategy** prevents convergence to local minima:
1. **Differential Evolution** globally explores the bounded 6D space, using per-factor Feller constraints and empirical $\sigma$ bounds.
2. **L-BFGS-B** then refines the DE solution to machine precision ($\text{ftol}=10^{-14}$).

In [ ]:
class TwoFactorCIR:
    """
    Two-factor additive CIR model with algebraic anchor state extraction.
    Short rate: r_t = x1_t + x2_t
    Each factor satisfies its own CIR SDE with independent Brownian motion.
    States are extracted via Cramer's rule using two observable anchor yields.
    """
    SHORT_COL   = 'ZC025YR'
    TARGET_COLS = ['ZC050YR','ZC075YR','ZC100YR','ZC200YR',
                   'ZC500YR','ZC1000YR','ZC2000YR','ZC3000YR']
    TARGET_TAUS = np.array([0.5, 0.75, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0])

    def __init__(self, train_df: pd.DataFrame):
        print('[2F-CIR] Initialising...')
        self._r  = train_df[self.SHORT_COL].values
        self._Y  = train_df[self.TARGET_COLS].values
        self._dt = 1.0 / 252.0
        self._n  = len(self._r)

        dr = np.diff(self._r)
        rm = np.maximum(self._r[:-1], 1e-8)
        # Divide by √2: per-factor σ assuming equal variance split
        self._emp_sig = float(np.sqrt(np.mean(dr**2 / (rm * self._dt)) / 2.0))
        print(f'  Per-factor σ̂ = {self._emp_sig:.6f}')

        self.params          = None
        self._ready          = False
        self._cached_states  = None

    # ── Affine factors (single factor) ───────────────────────────────────────
    @staticmethod
    def _AB(kappa, theta, sigma, tau):
        h    = np.sqrt(kappa**2 + 2.0*sigma**2)
        eht  = np.exp(h * tau)
        den  = 2.0*h + (kappa+h)*(eht - 1.0)
        B    = 2.0*(eht - 1.0) / den
        A    = (2.0*h*np.exp((kappa+h)*tau/2.0) / den) ** (2.0*kappa*theta/sigma**2)
        return A, B

    # ── Cramer's rule state extraction ────────────────────────────────────────
    def _extract(self, y1, y2, tau1, tau2, p):
        k1,th1,s1 = p['k1'],p['th1'],p['s1']
        k2,th2,s2 = p['k2'],p['th2'],p['s2']
        tau = np.array([tau1, tau2])

        A1,B1 = self._AB(k1,th1,s1,tau); A2,B2 = self._AB(k2,th2,s2,tau)
        Z1 = y1*tau1 + np.log(A1[0]*A2[0])
        Z2 = y2*tau2 + np.log(A1[1]*A2[1])

        det = B1[0]*B2[1] - B1[1]*B2[0]
        det = np.where(np.abs(det)<1e-10, 1e-10, det)

        x1  = (Z1*B2[1] - Z2*B2[0]) / det
        x2  = (B1[0]*Z2 - B1[1]*Z1) / det
        return np.maximum(x1,1e-8), np.maximum(x2,1e-8)

    # ── Training loss ─────────────────────────────────────────────────────────
    def _loss(self, raw):
        k1,th1,s1,k2,th2,s2 = raw
        if (any(v<=0 for v in [k1,th1,s1,k2,th2,s2])
                or 2*k1*th1<=s1**2 or 2*k2*th2<=s2**2):
            return 1e12

        p = dict(k1=k1,th1=th1,s1=s1,k2=k2,th2=th2,s2=s2)
        try:
            x1,x2 = self._extract(self._Y[:,0], self._Y[:,3],
                                   self.TARGET_TAUS[0], self.TARGET_TAUS[3], p)
        except Exception:
            return 1e12

        A1,B1 = self._AB(k1,th1,s1,self.TARGET_TAUS)
        A2,B2 = self._AB(k2,th2,s2,self.TARGET_TAUS)

        H  = np.column_stack([B1/self.TARGET_TAUS, B2/self.TARGET_TAUS])
        d  = -(np.log(A1)+np.log(A2)) / self.TARGET_TAUS
        st = np.column_stack([x1,x2])

        return float(np.mean((self._Y - st@H.T - d)**2))

    # ── Calibration ───────────────────────────────────────────────────────────
    def calibrate(self, popsize=10, tol=1e-4, maxiter=1000):
        s = self._emp_sig
        bounds = [(1e-3,5.0),(1e-3,0.20),(s*0.2,s*2.5)]*2

        print('[2F-CIR] Phase 1 — Differential Evolution...')
        de = differential_evolution(self._loss, bounds,
                                    strategy='best1bin', popsize=popsize,
                                    recombination=0.7, mutation=(0.5,1.0),
                                    tol=tol, seed=42, maxiter=maxiter,
                                    workers=-1, disp=True)
        print(f'  DE  loss: {de.fun:.8f}')

        print('[2F-CIR] Phase 2 — L-BFGS-B polish...')
        lbfgs = minimize(self._loss, de.x, method='L-BFGS-B', bounds=bounds,
                         options={'ftol':1e-14,'gtol':1e-9,'maxiter':3000})

        best = lbfgs if lbfgs.fun < de.fun else de
        k1,th1,s1,k2,th2,s2 = best.x
        self.params = dict(k1=k1,th1=th1,s1=s1,k2=k2,th2=th2,s2=s2)
        self._ready = True

        print('\n  Calibration complete')
        print(f'  Factor 1: κ={k1:.5f}  θ={th1*100:.4f}%  σ={s1:.5f}')
        print(f'  Factor 2: κ={k2:.5f}  θ={th2*100:.4f}%  σ={s2:.5f}')
        print(f'  Long-run rate: {(th1+th2)*100:.4f}%')
        self._cache_states()

    def _cache_states(self):
        p = self.params
        x1,x2 = self._extract(self._Y[:,0], self._Y[:,3],
                               self.TARGET_TAUS[0], self.TARGET_TAUS[3], p)
        self._cached_states = np.column_stack([x1,x2])

    # ── In-sample fit summary ─────────────────────────────────────────────────
    def fit_summary(self):
        p = self.params
        A1,B1 = self._AB(p['k1'],p['th1'],p['s1'],self.TARGET_TAUS)
        A2,B2 = self._AB(p['k2'],p['th2'],p['s2'],self.TARGET_TAUS)

        H  = np.column_stack([B1/self.TARGET_TAUS, B2/self.TARGET_TAUS])
        d  = -(np.log(A1)+np.log(A2)) / self.TARGET_TAUS
        pred = self._cached_states @ H.T + d
        rmse = np.sqrt(np.mean((self._Y - pred)**2, axis=0))

        print('\n  In-sample RMSE by tenor:')
        for tau,e in zip(self.TARGET_TAUS, rmse):
            print(f'    {tau:5.2f}Y → {e*1e4:6.2f} bps')
        print(f'    Overall → {rmse.mean()*1e4:.2f} bps')

    # ── Out-of-sample backtest ────────────────────────────────────────────────
    def backtest(self, test_df, anc1='ZC025YR', anc2='ZC200YR'):
        TMAP = {c: t for c, t in zip(['ZC025YR','ZC050YR','ZC075YR','ZC100YR',
                                       'ZC200YR','ZC500YR','ZC1000YR','ZC2000YR','ZC3000YR'],
                                      [0.25,0.5,0.75,1.0,2.0,5.0,10.0,20.0,30.0])}
        tau1, tau2 = TMAP[anc1], TMAP[anc2]
        eval_avail = {t: c for t, c in
                      {t: c for c, t in TMAP.items() if c in test_df.columns}.items()
                      if c not in (anc1, anc2)}

        ev_taus = np.array(sorted(eval_avail))
        ev_cols = [eval_avail[t] for t in ev_taus]

        Y_ev    = test_df[ev_cols].values
        y1 = test_df[anc1].values; y2 = test_df[anc2].values
        x1,x2 = self._extract(y1, y2, tau1, tau2, self.params)
        states = np.column_stack([x1,x2])

        p = self.params
        A1,B1 = self._AB(p['k1'],p['th1'],p['s1'],ev_taus)
        A2,B2 = self._AB(p['k2'],p['th2'],p['s2'],ev_taus)

        H  = np.column_stack([B1/ev_taus, B2/ev_taus])
        d  = -(np.log(A1)+np.log(A2)) / ev_taus
        Y_hat = states @ H.T + d

        r2_per  = {}; rmse_per = {}
        for i, tau in enumerate(ev_taus):
            yt,yp = Y_ev[:,i], Y_hat[:,i]
            r2_per[tau]   = float(1 - np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2))
            rmse_per[tau] = float(np.sqrt(np.mean((yt-yp)**2))*1e4)

        overall = float(np.mean(list(r2_per.values())))
        print(f'\n  Anchors: {anc1} ({tau1}yr) + {anc2} ({tau2}yr)')
        print(f'  Eval tenors: {ev_taus.tolist()}')
        print('\n  ' + '─'*50)
        for tau in ev_taus:
            bar = '█' * int(max(0,r2_per[tau])*20)
            print(f'  {tau:5.2f}yr │ {bar:<20} │ R²={r2_per[tau]:.4f}'
                  f'  RMSE={rmse_per[tau]:.1f}bps')
        print(f'  Overall R² = {overall:.4f}')

        return r2_per, overall, Y_hat, ev_taus

In [ ]:
tf_cir = TwoFactorCIR(train_df)
tf_cir.calibrate(popsize=10, tol=1e-4, maxiter=1000)
tf_cir.fit_summary()
r2_2f, overall_r2_2f, preds_2f, taus_2f = tf_cir.backtest(
    test_df, anc1='ZC025YR', anc2='ZC200YR')

### 6.4 Yield Curve Snapshots — Two-Factor CIR
The red dots mark the two anchor tenors: the model is **constrained to pass through these market quotes exactly** by construction (Cramer's rule). The green dashed line shows the predicted curve for all remaining unanchored tenors — these are genuine out-of-sample predictions, using only the anchor information.

In [ ]:
ANCHOR_COL1 = 'ZC025YR'; ANCHOR_COL2 = 'ZC200YR'
TAU1 = 0.25; TAU2 = 2.0

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('OOS Yield Curve — Two-Factor CIR (Algebraic Anchor Extraction)',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

ev_cols_2f = [c for c in TwoFactorCIR.TARGET_COLS
              if c in test_df.columns and c not in (ANCHOR_COL1, ANCHOR_COL2)]

for ax, date, si in zip(axes, snap_dates_t, snap_idx_t):
    row  = test_df.loc[date]
    ya1  = row[ANCHOR_COL1]; ya2 = row[ANCHOR_COL2]
    Y_ev = row[ev_cols_2f].values
    Y_pr = preds_2f[si]

    full_tau  = np.concatenate([[TAU1, TAU2], taus_2f])
    full_act  = np.concatenate([[ya1,  ya2],  Y_ev])
    full_pred = np.concatenate([[ya1,  ya2],  Y_pr])

    s_idx = np.argsort(full_tau)
    ax.plot(full_tau[s_idx], full_act[s_idx]  * 100,
            'o-',  color=PALETTE['actual'],    lw=2, ms=6, label='Market')
    ax.plot(full_tau[s_idx], full_pred[s_idx] * 100,
            's--', color=PALETTE['theta'],      lw=2, ms=5, label='2F-CIR')
    ax.scatter([TAU1, TAU2], [ya1*100, ya2*100],
               color=PALETTE['anchor'], s=120, zorder=5,
               edgecolors='k', label='Anchors')

    ax.set(title=str(date.date()), xlabel='Maturity (yr)', ylabel='Yield (%)')
    ax.legend(fontsize=8)

plt.tight_layout(rect=[0,0,1,0.95]); plt.show()

## 7. Critical Analysis — Model Limitations & Practical Implications
### 7.1 Structural Constraints of the Base CIR Model
**Single-factor correlation lock:** Because $r_t$ is driven by a single Brownian motion, every point on the yield curve is perfectly correlated. This prevents the model from reproducing yield curve twists (slope changes) or humps (curvature changes) — the dominant mode of real term-structure variation.

**Feller condition vs. volatility accuracy:** The Feller constraint $2\kappa\theta > \sigma^2$ is necessary for strict positivity, but it creates an adversarial optimisation trade-off in high-volatility, low-rate environments. Empirically, when the short rate sits near the Zero Lower Bound and realised volatility is elevated, satisfying Feller forces $\sigma$ below the market-implied level, degrading the diffusion component.

### 7.2 Extension Trade-offs
| Extension | Key Advantage | Primary Risk |
|-----------|--------------|--------------|
| **CIR++** | Perfect initial curve fit via $\Psi(\tau)$ | Look-back bias: static $\Psi$ anchored in historical averages; degrades after regime shifts |
| **AJD** | Captures discontinuous policy shocks | Identifiability: $\lambda$ and $\mu_J$ are statistically interchangeable in some regimes |
| **Two-Factor** | Independent short/long-end dynamics | Anchor sensitivity: errors in $\tau_1$ or $\tau_2$ quotes propagate linearly via Cramer's rule |

### 7.3 Implications for Fixed-Income Practitioners
**Model risk in relative-value trading:** CIR++ residuals can appear to be arbitrage signals when they are in fact systematic artefacts of spline interpolation. Trading against these residuals without accounting for the shift function's in-sample bias can produce systematic losses.

**Jump-model hedging:** A Poisson jump process implies discontinuous price paths, invalidating continuous Delta-hedging. A risk manager using the AJD model needs jump-exposure Greeks (analogous to Gamma for jump magnitude) in addition to standard rate duration.

**Regulatory preference:** Basel frameworks tend to favour simpler, more interpretable models. Despite its higher $R^2$, the AJD/Kalman combination may face model validation resistance due to the Kalman filter's latent-state opacity.

## 8. Theoretical Deep-Dive — Key Questions & Answers
**Q1: Why is Differential Evolution preferred over gradient-based solvers here?**
The CIR loss surface is highly non-convex under the Feller constraint. Standard BFGS and L-BFGS solvers, which follow the local gradient, reliably converge to economically implausible local minima (e.g. $\theta \approx 0$, $\sigma \rightarrow$ bound). Differential Evolution is a population-based global algorithm that simultaneously evaluates $15 \times d$ candidate solutions at every generation, efficiently mapping the full parameter space before descending. The subsequent L-BFGS polish then achieves machine-precision convergence at negligible extra cost.

**Q2: Under what conditions does the Feller condition bind in practice?**
The Feller condition $2\kappa\theta > \sigma^2$ is most likely to bind during **high-volatility, low-rate environments** — specifically when the central bank maintains near-zero short rates (suppressing $\theta$) while realised rate volatility (and hence the MLE estimate of $\sigma$) is elevated. In such regimes, satisfying Feller requires either an unrealistically high $\kappa$ (implying rates mean-revert to $\theta$ implausibly quickly) or an underestimated $\sigma$. The empirical sigma-anchor penalty introduced in this pipeline mitigates this by preventing the optimiser from driving $\sigma$ below the variance observed in data.

**Q3: What does the calibrated $\kappa$ imply about shock persistence?**
The mean-reverting half-life of an interest rate shock is $t_{1/2} = \ln(2)/\kappa$. A calibrated $\kappa \approx 0.15$ implies a half-life of approximately 4.6 years — consistent with the empirical observation that US interest rate regimes (e.g. Fed tightening cycles) persist for multi-year periods rather than reverting quickly.

**Q4: Why does the AJD model sometimes degrade in calm periods?**
The jump risk premium embedded in $\ln A_{\text{Jump}}(\tau)$ systematically inflates model-implied yields even on days when no jumps materialise. In calm market periods with realised $\lambda \approx 0$, this premium creates a structural upward bias. A time-varying jump intensity $\lambda(t)$ — estimated via regime-switching or rolling hazard-rate models — would resolve this at the cost of additional calibration complexity.

**Q5: What is the sensitivity of the Two-Factor model to anchor quality?**
Via Cramer's rule, the extracted state is a linear combination of the two anchor yields divided by the determinant $\text{det}(B)$. Near-collinearity of the loading functions $B_1(\tau)$ and $B_2(\tau)$ at the two anchor tenors makes the determinant small, amplifying any bid-offer noise or data-provider anomalies into large state estimation errors. The 3M/2Y anchor pair is specifically chosen because the two tenors have substantially different $B$ loading profiles, keeping $|\text{det}|$ robust.

## 9. Executive Summary
### Performance Comparison Across Models
| Model | OOS $R^2$ | OOS RMSE | Architecture Highlight |
|-------|-----------|----------|----------------------|
| Base CIR | 0.8755 | ~24 bps | Single-factor, 3 parameters, fully analytic |
| CIR++ | 0.9311 | ~18 bps | Deterministic shift corrects initial curve |
| AJD | 0.9342 | ~17 bps | Poisson jumps capture policy shock risk |
| Two-Factor | 0.9283 | ~19 bps | Independent short/long-end factor dynamics |

### Key Engineering Decisions
1. **Differential Evolution over gradient solvers** — mandatory for escaping Feller-constrained local minima on the CIR loss surface.
2. **Empirical sigma anchor** — prevents the optimiser exploiting the sigma-theta trade-off to produce an economically implausible volatility of near-zero.
3. **Rolling MAD filter** — robust to fat-finger outliers without inflating the rolling standard deviation that would normally be distorted by extreme events.
4. **Algebraic anchor inversion** — for the Two-Factor model, replaces the computationally expensive and tuning-sensitive Kalman Filter with a closed-form state extraction step accurate to floating-point precision.

### Recommended Architecture for Production
For an institutional fixed-income forecasting desk, the **AJD model with Extended Kalman Filter** provides the best out-of-sample accuracy while maintaining the analytical tractability of the affine class. CIR++ offers superior short-end precision for exotic options pricing but requires careful monitoring of the shift function $\Psi(\tau)$ for signs of regime obsolescence. The base CIR model remains the most robust choice for regulatory stress-testing scenarios where parameter interpretability is mandated.